<a href="https://colab.research.google.com/github/shahwaiz-9/Deep-Learning/blob/main/Complete_Analysis_on_time_series_data_using_lstm_.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [2]:
import kagglehub

# Download latest version
path = kagglehub.dataset_download("msafi04/predict-mortality-of-icu-patients-physionet")

print("Path to dataset files:", path)

Using Colab cache for faster access to the 'predict-mortality-of-icu-patients-physionet' dataset.
Path to dataset files: /kaggle/input/predict-mortality-of-icu-patients-physionet


##**Preparing Data**

In [3]:
import pandas as pd
import os

# Define base path from the previously downloaded dataset
base_path = path

# Define paths for outcome file and set-a folder
outcome_file_path = os.path.join(base_path, 'Outcomes-a.txt')
set_a_folder_path = os.path.join(base_path, 'set-a', 'set-a')

print(f"Outcome file path: {outcome_file_path}")
print(f"Set-A folder path: {set_a_folder_path}")

Outcome file path: /kaggle/input/predict-mortality-of-icu-patients-physionet/Outcomes-a.txt
Set-A folder path: /kaggle/input/predict-mortality-of-icu-patients-physionet/set-a/set-a


In [4]:
print(f"Contents of {base_path}:")
print(os.listdir(base_path))

Contents of /kaggle/input/predict-mortality-of-icu-patients-physionet:
['set-a', 'Outcomes-a.txt']


First, let's load the `outcome-a.txt` file, which typically contains patient IDs and their outcomes or other high-level details.

In [5]:
# Load outcome-a.txt
try:
    outcome_df = pd.read_csv(outcome_file_path)
    print("Outcome-A data loaded successfully.")
    display(outcome_df.head())
except FileNotFoundError:
    print(f"Error: outcome-a.txt not found at {outcome_file_path}")
    outcome_df = None
except Exception as e:
    print(f"An error occurred while loading outcome-a.txt: {e}")
    outcome_df = None

Outcome-A data loaded successfully.


,RecordID,SAPS-I,SOFA,Length_of_stay,Survival,In-hospital_death
0,132539,6,1,5,-1,0
1,132540,16,8,8,-1,0
2,132541,21,11,19,-1,0
3,132543,7,1,9,575,0
4,132545,17,2,4,918,0


Now, we will process the individual patient data files from the `set-a` folder as per your provided logic. This involves reading each file, extracting relevant information, pivoting the data to have hours as rows and parameters as columns, and ensuring all 48 hours are present.

In [34]:
outcome_df['Survival'].value_counts()

,count
Survival,
-1,2526
2,57
3,52
5,45
4,39
...,...
543,1
510,1
540,1


In [6]:
all_patient_data = []

# Check if the folder exists
if not os.path.exists(set_a_folder_path):
    print(f"Error: The folder {set_a_folder_path} does not exist.")
else:
    for filename in os.listdir(set_a_folder_path):
        if filename.endswith(".txt"):
            filepath = os.path.join(set_a_folder_path, filename)

            try:

                df = pd.read_csv(filepath, sep=',')

                record_id_row = df[df['Parameter'] == 'RecordID']
                if not record_id_row.empty:
                    record_id = int(record_id_row['Value'].values[0])
                else:

                    record_id = int(filename.split('.')[0])

                # Convert Time to Hour
                df['Hour'] = df['Time'].apply(lambda x: int(x.split(':')[0]))


                df['Value'] = pd.to_numeric(df['Value'], errors='coerce')

                pivoted = df.pivot_table(index='Hour', columns='Parameter', values='Value', aggfunc='mean')

                # Ensure we have all 48 hours (0 to 47)
                pivoted = pivoted.reindex(range(48))

                # Add the PatientID column back for identification
                pivoted['PatientID'] = record_id

                all_patient_data.append(pivoted)
            except Exception as e:
                print(f"Error processing file {filename}: {e}")

    # Combine everything into one massive DataFrame
    if all_patient_data:
        master_df = pd.concat(all_patient_data).reset_index()
        print("Master DataFrame created successfully.")
        display(master_df.head())
    else:
        print("No patient data was processed.")
        master_df = pd.DataFrame()

Master DataFrame created successfully.


Parameter,Hour,ALT,Age,Albumin,BUN,Cholesterol,Creatinine,GCS,Gender,Glucose,...,PaO2,SysABP,pH,ALP,AST,Bilirubin,SaO2,RespRate,TroponinT,TroponinI
0,0,NaN,66.0,NaN,NaN,NaN,NaN,NaN,1.0,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,1,NaN,NaN,NaN,NaN,NaN,NaN,15.0,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,2,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,3,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,4,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


Finally, let's export the combined `master_df` to a CSV file for easy reuse.

In [7]:
if not master_df.empty:
    output_csv_path = 'processed_patient_data.csv'
    master_df.to_csv(output_csv_path, index=False)
    print(f"Master DataFrame exported to '{output_csv_path}'")
else:
    print("Master DataFrame is empty, skipping CSV export.")

Master DataFrame exported to 'processed_patient_data.csv'


##**Overview**

In [8]:
master_df.columns

Index(['Hour', 'ALT', 'Age', 'Albumin', 'BUN', 'Cholesterol', 'Creatinine',
       'GCS', 'Gender', 'Glucose', 'HCO3', 'HCT', 'HR', 'Height', 'ICUType',
       'K', 'Mg', 'NIDiasABP', 'NIMAP', 'NISysABP', 'Na', 'Platelets',
       'RecordID', 'Temp', 'Urine', 'WBC', 'Weight', 'PatientID', 'DiasABP',
       'FiO2', 'Lactate', 'MAP', 'MechVent', 'PaCO2', 'PaO2', 'SysABP', 'pH',
       'ALP', 'AST', 'Bilirubin', 'SaO2', 'RespRate', 'TroponinT',
       'TroponinI'],
      dtype='object', name='Parameter')

In [9]:
master_df.isnull().sum()

,0
Parameter,
Hour,0
ALT,188829
Age,188000
Albumin,189648
BUN,178113
Cholesterol,191685
Creatinine,178047
GCS,130948
Gender,188000


In [10]:
master_df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 192000 entries, 0 to 191999
Data columns (total 44 columns):
 #   Column       Non-Null Count   Dtype  
---  ------       --------------   -----  
 0   Hour         192000 non-null  int64  
 1   ALT          3171 non-null    float64
 2   Age          4000 non-null    float64
 3   Albumin      2352 non-null    float64
 4   BUN          13887 non-null   float64
 5   Cholesterol  315 non-null     float64
 6   Creatinine   13953 non-null   float64
 7   GCS          61052 non-null   float64
 8   Gender       4000 non-null    float64
 9   Glucose      12988 non-null   float64
 10  HCO3         13582 non-null   float64
 11  HCT          18169 non-null   float64
 12  HR           172883 non-null  float64
 13  Height       4000 non-null    float64
 14  ICUType      4000 non-null    float64
 15  K            14406 non-null   float64
 16  Mg           13568 non-null   float64
 17  NIDiasABP    80761 non-null   float64
 18  NIMAP        79748 non-n

In [11]:


null_ratios = master_df.isnull().sum() / len(master_df)

raw_null_ratios = null_ratios[~null_ratios.index.str.contains('_measured')]

# Print the report
print("--- Missing Data Report ---")
for col, ratio in raw_null_ratios.items():
    print(f"{col:25} | {ratio:.2%} missing")


very_sparse_cols = raw_null_ratios[raw_null_ratios > 0.80].index.tolist()
medium_sparse_cols = raw_null_ratios[(raw_null_ratios <= 0.80) & (raw_null_ratios > 0.20)].index.tolist()
common_cols = raw_null_ratios[raw_null_ratios <= 0.20].index.tolist()

print(f"\nDropping {len(very_sparse_cols)} columns (Threshold > 80%)")

--- Missing Data Report ---
Hour                      | 0.00% missing
ALT                       | 98.35% missing
Age                       | 97.92% missing
Albumin                   | 98.78% missing
BUN                       | 92.77% missing
Cholesterol               | 99.84% missing
Creatinine                | 92.73% missing
GCS                       | 68.20% missing
Gender                    | 97.92% missing
Glucose                   | 93.24% missing
HCO3                      | 92.93% missing
HCT                       | 90.54% missing
HR                        | 9.96% missing
Height                    | 97.92% missing
ICUType                   | 97.92% missing
K                         | 92.50% missing
Mg                        | 92.93% missing
NIDiasABP                 | 57.94% missing
NIMAP                     | 58.46% missing
NISysABP                  | 57.89% missing
Na                        | 92.95% missing
Platelets                 | 92.70% missing
RecordID                  | 

Many null values are there. Basically the approach is to classify columns into 2 parts
1. Static columns ( Age )
2. Continuous columns ( Heart rate etc ). <br>
<br>
note: Since there are columns which are not static nor complete continous. To tackle this we will make them static by talking valueable information from the data like min, max , average etc

##**Static Columns Cleaning**

In [12]:
# Making 2 dataframes from master-df


sparse_cols = [
    # Metabolic & Electrolytes
    'Glucose', 'BUN', 'Creatinine', 'Na', 'K', 'Mg', 'HCO3', 'HCT',

    # Enzymes & Liver Function
    'ALP', 'ALT', 'AST', 'Bilirubin', 'Albumin', 'Lactate', 'Cholesterol',

    # Cardiac & Blood Markers
    'TroponinI', 'TroponinT', 'Platelets', 'WBC',

    # Blood Gas & Respiratory (Sparse in many patients)
    'pH', 'PaCO2', 'PaO2', 'SaO2', 'FiO2', 'MechVent'
]


static_cols = ['Age', 'Gender', 'Height', 'ICUType']


patient_summary = master_df.groupby('PatientID').agg({
    **{col: ['first', 'last', 'mean', 'min', 'max'] for col in sparse_cols},
    **{col: 'first' for col in static_cols}
})



patient_summary.columns = ['_'.join(col).strip() if isinstance(col, tuple) else col for col in patient_summary.columns]

In [13]:
patient_summary.head(5)

,Glucose_first,Glucose_last,Glucose_mean,Glucose_min,Glucose_max,BUN_first,BUN_last,BUN_mean,BUN_min,BUN_max,...,FiO2_max,MechVent_first,MechVent_last,MechVent_mean,MechVent_min,MechVent_max,Age_first,Gender_first,Height_first,ICUType_first
PatientID,,,,,,,,,,,,,,,,,,,,,
132539,205.0,115.0,160.000000,115.0,205.0,13.0,8.0,10.500000,8.0,13.0,...,NaN,NaN,NaN,NaN,NaN,NaN,54.0,0.0,-1.0,4.0
132540,105.0,146.0,125.500000,105.0,146.0,16.0,21.0,18.333333,16.0,21.0,...,0.75,1.0,1.0,1.0,1.0,1.0,76.0,1.0,175.3,2.0
132541,141.0,143.0,134.333333,119.0,143.0,8.0,3.0,4.666667,3.0,8.0,...,1.00,1.0,1.0,1.0,1.0,1.0,44.0,0.0,-1.0,3.0
132543,129.0,117.0,117.333333,106.0,129.0,23.0,10.0,17.666667,10.0,23.0,...,NaN,NaN,NaN,NaN,NaN,NaN,68.0,1.0,180.3,3.0
132545,113.0,92.0,102.500000,92.0,113.0,45.0,25.0,35.000000,25.0,45.0,...,NaN,NaN,NaN,NaN,NaN,NaN,88.0,0.0,-1.0,3.0


In [14]:
patient_summary['Cholesterol_first'].unique()

array([ nan, 212.,  84., 181., 151., 111.,  92., 205., 235., 152., 161.,
       227., 174., 103., 170., 134., 215.,  99., 133., 167., 189., 101.,
       155., 216.,  85., 159., 128., 136., 157., 192., 263., 104., 154.,
       139., 122., 147., 171., 164., 114.,  95., 144., 163., 188., 185.,
       140., 330., 160., 127., 237., 184., 187., 149., 116., 123., 121.,
       124., 166., 190.,  96.,  91., 243., 117., 175., 213., 129., 198.,
       172., 258., 168., 162., 150., 135.,  88., 178., 222., 143., 250.,
       109., 100., 131., 201., 207.,  75., 126., 113., 194.,  59., 169.,
       125., 244., 148., 183., 307., 115., 206., 180., 119., 230., 199.,
       179., 145., 141.,  94.,  73., 158., 217., 138., 130., 297., 211.,
       220., 202., 165., 195., 245., 224., 197., 289., 146., 209., 177.,
       200., 102., 112., 105., 218., 204., 142.,  28., 137., 153.,  90.,
       173.,  86., 191., 118.,  40.,  81., 208.,  68., 132.])

In [15]:
patient_summary.info()

<class 'pandas.core.frame.DataFrame'>
Index: 4000 entries, 132539 to 142673
Columns: 129 entries, Glucose_first to ICUType_first
dtypes: float64(129)
memory usage: 4.0 MB


In [16]:


null_ratios = patient_summary.isnull().sum() / len(patient_summary)

raw_null_ratios = null_ratios[~null_ratios.index.str.contains('_measured')]

# Print the report
print("--- Missing Data Report ---")
for col, ratio in raw_null_ratios.items():
    print(f"{col:25} | {ratio:.2%} missing")


very_sparse_cols = raw_null_ratios[raw_null_ratios > 0.80].index.tolist()
medium_sparse_cols = raw_null_ratios[(raw_null_ratios <= 0.80) & (raw_null_ratios > 0.20)].index.tolist()
common_cols = raw_null_ratios[raw_null_ratios <= 0.20].index.tolist()

print(f"\nDropping {len(very_sparse_cols)} columns (Threshold > 80%)")

--- Missing Data Report ---
Glucose_first             | 2.83% missing
Glucose_last              | 2.83% missing
Glucose_mean              | 2.83% missing
Glucose_min               | 2.83% missing
Glucose_max               | 2.83% missing
BUN_first                 | 1.60% missing
BUN_last                  | 1.60% missing
BUN_mean                  | 1.60% missing
BUN_min                   | 1.60% missing
BUN_max                   | 1.60% missing
Creatinine_first          | 1.60% missing
Creatinine_last           | 1.60% missing
Creatinine_mean           | 1.60% missing
Creatinine_min            | 1.60% missing
Creatinine_max            | 1.60% missing
Na_first                  | 1.88% missing
Na_last                   | 1.88% missing
Na_mean                   | 1.88% missing
Na_min                    | 1.88% missing
Na_max                    | 1.88% missing
K_first                   | 2.40% missing
K_last                    | 2.40% missing
K_mean                    | 2.40% missing
K_min 

Basically there is no point of filling columns having missing values greater then 80% this will add extra noise for the model. Instead we will have column which will tells weather that particular test was taken or not.

In [17]:
# 1. Very Healthy / Common (Minimal missingness)
common = [
    'Glucose', 'BUN', 'Creatinine', 'Na', 'K', 'Mg',
    'HCO3', 'HCT', 'Platelets', 'WBC'
]

# 2. The Danger Zone (Significant missingness, but keep the average signal)
danger_zone = [
    'Lactate', 'Bilirubin', 'ALP', 'ALT', 'AST', 'Albumin',
    'pH', 'PaCO2', 'PaO2', 'SaO2', 'FiO2', 'MechVent'
]

# 3. The Ghost Tier (Now including Troponin due to 75% threshold)
ghosts = ['Cholesterol', 'TroponinI', 'TroponinT']

def final_medical_cleanup(df):
    # Process Danger Zone: Keep Mean + Flag, Drop First/Last/Min/Max
    for lab in danger_zone:
        df[f'{lab}_measured'] = df[f'{lab}_mean'].notnull().astype(int)
        cols_to_drop = [f'{lab}_first', f'{lab}_last', f'{lab}_min', f'{lab}_max']
        df.drop(columns=cols_to_drop, errors='ignore', inplace=True)

    # Process Ghosts: Keep Flag Only, Drop ALL Numerics
    for lab in ghosts:
        df[f'{lab}_measured'] = df[f'{lab}_mean'].notnull().astype(int)
        cols_to_drop = [f'{lab}_first', f'{lab}_last', f'{lab}_mean', f'{lab}_min', f'{lab}_max']
        df.drop(columns=cols_to_drop, errors='ignore', inplace=True)

    # --- Grouped Imputation Logic ---
    # Create Age Bins for peer-group comparison
    df['Age_Group'] = pd.cut(df['Age_first'], bins=[0, 30, 50, 70, 120], labels=[1, 2, 3, 4])

    # Identify numeric columns for imputation
    numeric_cols = df.select_dtypes(include=['number']).columns

    # Fill based on ICU Type and Age Peer Group
    df[numeric_cols] = df.groupby(['ICUType_first', 'Age_Group'], group_keys=False)[numeric_cols].transform(
        lambda x: x.fillna(x.median())
    )

    # Final Safety Net: Global Median for extremely rare cases
    df = df.fillna(df.median(numeric_only=True))

    # Clean up helper column
    df.drop(columns=['Age_Group'], inplace=True)

    return df

# Execute the final pruning
final_df = final_medical_cleanup(patient_summary)

/tmp/ipykernel_1800/3206902828.py:37: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  df[numeric_cols] = df.groupby(['ICUType_first', 'Age_Group'], group_keys=False)[numeric_cols].transform(


In [18]:
final_df.head(4)

,Glucose_first,Glucose_last,Glucose_mean,Glucose_min,Glucose_max,BUN_first,BUN_last,BUN_mean,BUN_min,BUN_max,...,Albumin_measured,pH_measured,PaCO2_measured,PaO2_measured,SaO2_measured,FiO2_measured,MechVent_measured,Cholesterol_measured,TroponinI_measured,TroponinT_measured
PatientID,,,,,,,,,,,,,,,,,,,,,
132539,205.0,115.0,160.000000,115.0,205.0,13.0,8.0,10.500000,8.0,13.0,...,0,0,0,0,0,0,0,0,0,0
132540,105.0,146.0,125.500000,105.0,146.0,16.0,21.0,18.333333,16.0,21.0,...,0,1,1,1,1,1,1,0,0,0
132541,141.0,143.0,134.333333,119.0,143.0,8.0,3.0,4.666667,3.0,8.0,...,1,1,1,1,1,1,1,0,0,0
132543,129.0,117.0,117.333333,106.0,129.0,23.0,10.0,17.666667,10.0,23.0,...,1,0,0,0,0,0,0,0,0,0


In [19]:
final_df.isnull().sum()

,0
Glucose_first,0
Glucose_last,0
Glucose_mean,0
Glucose_min,0
Glucose_max,0
...,...
FiO2_measured,0
MechVent_measured,0
Cholesterol_measured,0
TroponinI_measured,0


##**Countinuous Columns Cleaning**

In [20]:
# These columns will form the 48-step sequence for the LSTM
seq_cols = [
    'HR', 'Urine', 'SysABP', 'DiasABP', 'MAP',
    'Weight', 'NIDiasABP', 'NISysABP', 'NIMAP',
    'Temp', 'GCS', 'RespRate'
]

In [21]:

# Forward fill

def process_continuous_sequences(master_df, seq_cols):
    # 1. Create a copy of the sequential data
    seq_df = master_df[['PatientID', 'Hour'] + seq_cols].copy()

    # 2. Ensure chronological order
    seq_df = seq_df.sort_values(['PatientID', 'Hour'])

    # 3. Forward Fill: Carry the last known value forward
    seq_df[seq_cols] = seq_df.groupby('PatientID')[seq_cols].ffill()

    # 4. Backward Fill: Handle cases where the first few hours are empty
    seq_df[seq_cols] = seq_df.groupby('PatientID')[seq_cols].bfill()

    # 5. Final Median Fill: For parameters NEVER measured for a specific patient
    seq_df[seq_cols] = seq_df[seq_cols].fillna(seq_df[seq_cols].median())

    return seq_df

clean_seq_df = process_continuous_sequences(master_df, seq_cols)

In [22]:
from sklearn.preprocessing import StandardScaler

scaler = StandardScaler()
clean_seq_df[seq_cols] = scaler.fit_transform(clean_seq_df[seq_cols])

In [40]:
final_df = scaler.fit_transform(final_df)

In [36]:
y = outcome_df['In-hospital_death'].values

**Modelling**

In [23]:
def create_3d_cube(df, seq_cols):
    raw_values = df[seq_cols].values

    X_sequential = raw_values.reshape(-1, 48, len(seq_cols))

    return X_sequential

X_seq = create_3d_cube(clean_seq_df, seq_cols)
print(f"Final LSTM Input Shape: {X_seq.shape}")

Final LSTM Input Shape: (4000, 48, 12)


In [30]:
from tensorflow.keras.models import Model
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import LSTM, Dense, Dropout, Input, concatenate

In [27]:
X_seq.shape[2]

12

In [52]:
# handling imbalance dataset

from sklearn.utils import class_weight
import numpy as np
# Calculate weights: Higher weight for the minority class

weights = class_weight.compute_class_weight(
    class_weight='balanced',
    classes=np.unique(y),
    y=y
)

class_weight_dict = {0: weights[0], 1: weights[1]}

print(f"Class Weights: {class_weight_dict}")


Class Weights: {0: np.float64(0.5803830528148578), 1: np.float64(3.6101083032490973)}


In [59]:
def build_model():


  lstm_input = Input(shape=(48, X_seq.shape[2]))
  x = LSTM(64, return_sequences=False)(lstm_input)
  x = Dropout(0.3)(x)
  x = Dense(32, activation='relu')(x)



  static_input = Input(shape=(final_df.shape[1],), name='Static_Input')
  x1 = Dense(128, activation='relu')(static_input)
  x1 = Dropout(0.3)(x1)
  x1 = Dense(64, activation='relu')(x1)
  x1 = Dropout(0.3)(x1)
  x1 = Dense(32, activation='relu')(x1)

  combined = concatenate([x, x1])
  z = Dense(32, activation='relu')(combined)
  z = Dropout(0.3)(z)
  output = Dense(1, activation='sigmoid', name='Mortality_Prediction')(z)

  model = Model(inputs = [lstm_input, static_input], outputs = output)

  model.compile(optimizer='adam', loss='binary_crossentropy', metrics=['accuracy'])

  model.summary()


  return model


In [60]:
lstm = build_model()

Model: "functional_5"

┏━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)        ┃ Output Shape      ┃    Param # ┃ Connected to      ┃
┡━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━┩
│ Static_Input        │ (None, 81)        │          0 │ -                 │
│ (InputLayer)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dense_27 (Dense)    │ (None, 128)       │     10,496 │ Static_Input[0][… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ input_layer_7       │ (None, 48, 12)    │          0 │ -                 │
│ (InputLayer)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dropout_20          │ (None, 128)       │          0 │ dense_27[0][0]    │
│ (Dropout)           │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ lstm_7 (LSTM)       │ (None, 64)        │     19,712 │ input_layer_7[0]… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dense_28 (Dense)    │ (None, 64)        │      8,256 │ dropout_20[0][0]  │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dropout_19          │ (None, 64)        │          0 │ lstm_7[0][0]      │
│ (Dropout)           │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dropout_21          │ (None, 64)        │          0 │ dense_28[0][0]    │
│ (Dropout)           │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dense_26 (Dense)    │ (None, 32)        │      2,080 │ dropout_19[0][0]  │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dense_29 (Dense)    │ (None, 32)        │      2,080 │ dropout_21[0][0]  │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ concatenate_5       │ (None, 64)        │          0 │ dense_26[0][0],   │
│ (Concatenate)       │                   │            │ dense_29[0][0]    │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dense_30 (Dense)    │ (None, 32)        │      2,080 │ concatenate_5[0]… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dropout_22          │ (None, 32)        │          0 │ dense_30[0][0]    │
│ (Dropout)           │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ Mortality_Predicti… │ (None, 1)         │         33 │ dropout_22[0][0]  │
│ (Dense)             │                   │            │                   │
└─────────────────────┴───────────────────┴────────────┴───────────────────┘

 Total params: 44,737 (174.75 KB)

 Trainable params: 44,737 (174.75 KB)

 Non-trainable params: 0 (0.00 B)

In [61]:

history = lstm.fit(
    x=[X_seq, final_df], # Note the list of inputs
    y=y,
    epochs=10,
    class_weight = class_weight_dict,
    batch_size=8,
    validation_split=0.2, # Holds 20% for testing during training
    verbose=1
)

Epoch 1/10
400/400 ━━━━━━━━━━━━━━━━━━━━ 11s 18ms/step - accuracy: 0.6694 - loss: 0.6000 - val_accuracy: 0.6762 - val_loss: 0.5645
Epoch 2/10
400/400 ━━━━━━━━━━━━━━━━━━━━ 9s 15ms/step - accuracy: 0.7241 - loss: 0.5210 - val_accuracy: 0.7800 - val_loss: 0.4386
Epoch 3/10
400/400 ━━━━━━━━━━━━━━━━━━━━ 7s 17ms/step - accuracy: 0.7459 - loss: 0.4827 - val_accuracy: 0.7113 - val_loss: 0.5224
Epoch 4/10
400/400 ━━━━━━━━━━━━━━━━━━━━ 7s 17ms/step - accuracy: 0.7441 - loss: 0.4564 - val_accuracy: 0.7362 - val_loss: 0.4559
Epoch 5/10
400/400 ━━━━━━━━━━━━━━━━━━━━ 8s 21ms/step - accuracy: 0.7663 - loss: 0.4291 - val_accuracy: 0.7837 - val_loss: 0.4023
Epoch 6/10
400/400 ━━━━━━━━━━━━━━━━━━━━ 9s 18ms/step - accuracy: 0.7675 - loss: 0.4325 - val_accuracy: 0.7237 - val_loss: 0.4820
Epoch 7/10
400/400 ━━━━━━━━━━━━━━━━━━━━ 6s 15ms/step - accuracy: 0.7638 - loss: 0.4038 - val_accuracy: 0.7188 - val_loss: 0.5040
Epoch 8/10
400/400 ━━━━━━━━━━━━━━━━━━━━ 12s 19ms/step - accuracy: 0.7759 - loss: 0.4049 - val_ac

## Model Evaluation

In [63]:
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, roc_auc_score

X_seq_train, X_seq_val, final_df_train, final_df_val, y_train, y_val = train_test_split(
    X_seq, final_df, y, test_size=0.2, random_state=42 # Using a fixed random_state for reproducibility
)


y_pred_train_proba = lstm.predict([X_seq_train, final_df_train])
y_pred_val_proba = lstm.predict([X_seq_val, final_df_val])


y_pred_train = (y_pred_train_proba > 0.5).astype(int)
y_pred_val = (y_pred_val_proba > 0.5).astype(int)

print("--- Training Set Metrics ---")
print(f"Accuracy: {accuracy_score(y_train, y_pred_train):.4f}")
print(f"Precision: {precision_score(y_train, y_pred_train):.4f}")
print(f"Recall: {recall_score(y_train, y_pred_train):.4f}")
print(f"F1-Score: {f1_score(y_train, y_pred_train):.4f}")
print(f"AUC: {roc_auc_score(y_train, y_pred_train_proba):.4f}")

print("\n--- Validation Set Metrics ---")
print(f"Accuracy: {accuracy_score(y_val, y_pred_val):.4f}")
print(f"Precision: {precision_score(y_val, y_pred_val):.4f}")
print(f"Recall: {recall_score(y_val, y_pred_val):.4f}")
print(f"F1-Score: {f1_score(y_val, y_pred_val):.4f}")
print(f"AUC: {roc_auc_score(y_val, y_pred_val_proba):.4f}")

100/100 ━━━━━━━━━━━━━━━━━━━━ 1s 9ms/step
25/25 ━━━━━━━━━━━━━━━━━━━━ 0s 9ms/step
--- Training Set Metrics ---
Accuracy: 0.7978
Precision: 0.3955
Recall: 0.9353
F1-Score: 0.5559
AUC: 0.9308

--- Validation Set Metrics ---
Accuracy: 0.7812
Precision: 0.3977
Recall: 0.8678
F1-Score: 0.5455
AUC: 0.9095
